# CrackScan: train, evaluate and run the website (Google Colab / Kaggle)
**Before you start**
- **Colab**: Runtime -> Change runtime type -> **T4 GPU**.
- **Kaggle**: right panel -> Accelerator **GPU T4 x2/P100** and **Internet ON** (needed to clone the dataset and download pretrained weights).
- Have `crack_detection.zip` ready (Colab: you will upload it; Kaggle: upload the folder as a *Dataset* and add it to this notebook).

Run the cells top to bottom. Every cell prints what it did.

In [ ]:
# 1) Put the project in the working directory
import os, sys, glob, shutil, zipfile
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_COLAB:
    from google.colab import files
    print("Select crack_detection.zip ...")
    files.upload()
    zipfile.ZipFile("crack_detection.zip").extractall(".")
    os.chdir("/content/crack_detection")
elif ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/app.py", recursive=True)
    assert hits, "Add the project folder as a Kaggle Dataset to this notebook first."
    shutil.copytree(os.path.dirname(hits[0]), "/kaggle/working/crack_detection", dirs_exist_ok=True)
    os.chdir("/kaggle/working/crack_detection")
print("working dir:", os.getcwd()); print(os.listdir("."))

In [ ]:
# 2) Dependencies (torch / torchvision / opencv are already installed on Colab and Kaggle)
!pip -q install "streamlit>=1.40,<1.50" scikit-image
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 3) Data (Objective 2)
DeepCrack gives pixel masks (segmentation). We also cut **128 px patches** from it for the classification task, so one download covers everything.
> If the clone below fails, download DeepCrack (or the merged *crack segmentation* dataset) from Kaggle/GitHub, put it under `data/DeepCrack/`, and re-run only the `data_prep` lines. The pairing code finds image/mask folders automatically (`train_img/train_lab`, `images/masks`, `*_mask.png`, ...).

In [ ]:
!git clone --depth 1 https://github.com/yhlleo/DeepCrack.git data/DeepCrack
!python -m src.data_prep seg --root data/DeepCrack
!python -m src.data_prep cls-from-seg --seg_csv data/seg_split.csv --patch 128

## 4) Segmentation: baseline vs enhancement (Objectives 3 & 4)
Baseline = plain U-Net with BCE loss. Enhanced = **Attention U-Net + BCE+Dice loss**. Same data, same epochs, so the comparison is fair.
Roughly 5-10 minutes each on a T4.

In [ ]:
!python -m src.train_seg --model unet --loss bce --epochs 40
!python -m src.train_seg --model attention_unet --loss bce_dice --epochs 40

Optional ablation (shows which part of the enhancement helps): attention only, with the baseline loss.

In [ ]:
# !python -m src.train_seg --model attention_unet --loss bce --epochs 40 --tag attn_only

## 5) Classification baselines (Objective 3)

In [ ]:
!python -m src.train_cls --model resnet18 --epochs 6
!python -m src.train_cls --model efficientnet_b0 --epochs 6
# !python -m src.train_cls --model mobilenet_v3_small --epochs 6

## 6) Optional: YOLOv8 detection (Objective 3)
Boxes are derived from the masks. Skip this cell if time is short; the website already shows mask-derived boxes.

In [ ]:
# !pip -q install ultralytics
# !python -m src.make_yolo_dataset --seg_csv data/seg_split.csv --out data/yolo
# from ultralytics import YOLO; import shutil, json, pathlib
# YOLO("yolov8n.pt").train(data="data/yolo/data.yaml", epochs=40, imgsz=416, batch=16, project="runs", name="crack", exist_ok=True)
# shutil.copy("runs/crack/weights/best.pt", "weights/yolo_crack.pt")
# r = YOLO("weights/yolo_crack.pt").val(data="data/yolo/data.yaml", split="test")
# pathlib.Path("results/det_yolo_metrics.json").write_text(json.dumps({"label": "YOLOv8n", "map50": float(r.box.map50),
#     "map50_95": float(r.box.map), "precision": float(r.box.mp), "recall": float(r.box.mr)}))

## 7) Robustness evaluation (Objective 6)
Perturbations (dark, bright, low contrast, shadow, blur, noise, JPEG) x 3 severities, thin-vs-thick cracks, and optionally an **unseen dataset**.
For the unseen-surface test, prepare a second dataset (e.g. CRACK500) with
`!python -m src.data_prep seg --root data/CRACK500 --out_csv data/crack500_split.csv` and add `--cross_csv data/crack500_split.csv` below.

In [ ]:
!python -m src.evaluate --models unet attention_unet

## 8) Package results (download `crack_detection_trained.zip`)

In [ ]:
!python -m src.export_weights
import shutil, os
os.makedirs("/tmp/pkg", exist_ok=True)
shutil.make_archive("/tmp/pkg/crack_detection_trained", "zip", ".", "weights")
shutil.make_archive("/tmp/pkg/crack_detection_results", "zip", ".", "results")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/tmp/pkg/crack_detection_trained.zip"); files.download("/tmp/pkg/crack_detection_results.zip")
else:
    shutil.copy("/tmp/pkg/crack_detection_trained.zip", "/kaggle/working/"); shutil.copy("/tmp/pkg/crack_detection_results.zip", "/kaggle/working/")
    print("Find both zips in the Output tab (right panel).")

## 9) Run the website from this notebook (temporary public link)
A free Cloudflare quick tunnel gives you a `https://....trycloudflare.com` link while this cell keeps running. Use it to test before deploying to Streamlit Cloud.

In [ ]:
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
log = open("cf.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=log, stderr=log)
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        print("OPEN THIS LINK ->", m.group(0)); break
else:
    print("No link yet; print(open('cf.log').read()) to debug")